<a href="https://colab.research.google.com/github/LaraBBrum/automata-vs-ml-mbt/blob/master/ESBMC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Verificação formal (ESBMC) — AALpy, Agilkia e ML

Fluxo: instalar o ESBMC → subir os esbmc_*.json exportados pela subseção 12.1 do ML.ipynb (com ml_sequences e mef) →
gerar e rodar a verificação das **sequências** (AALpy, Agilkia, ML) → verificar a **máquina de estados** aprendida (MEF).

Os JSON devem ser os da última rodada do ML.ipynb; a célula de upload apaga os antigos do ambiente e normaliza os nomes.

In [ ]:
# 1. Instala dependências necessárias no sistema
!apt-get update -qq --fix-missing
!apt-get install -y clang libclang-dev z3 curl -qq

# 2. Baixa a versão mais recente diretamente do repositório via API do GitHub
!curl -s https://api.github.com/repos/esbmc/esbmc/releases/latest \
  | grep "browser_download_url.*esbmc-linux.zip" \
  | cut -d : -f 2,3 \
  | tr -d \" \
  | head -n 1 \
  | wget -i - -O esbmc-linux.zip

# 3. Extrai e instala o binário
!unzip -q -o esbmc-linux.zip -d /tmp/esbmc_out || tar -xzf esbmc-linux.zip -C /tmp/esbmc_out
!find /tmp/esbmc_out -name "esbmc" -exec cp {} /usr/local/bin/ \;
!chmod +x /usr/local/bin/esbmc

# 4. Confirma a instalação
!esbmc --version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package python3-pygments.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../00-python3-pygments_2.17.2+dfsg-1_all.deb ...
Unpacking python3-pygments (2.17.2+dfsg-1) ...
Selecting previously unselected package python3-yaml.
Preparing to unpack .../01-python3-yaml_6.0.1-2build2_amd64.deb ...
Unpacking python3-yaml (6.0.1-2build2) ...
Selecting previously unselected package libllvm18:amd64.
Preparing to unpack .../02-libllvm18_1%3a18.1.3-1ubuntu1_amd64.deb ...
Unpacking libllvm18:amd64 (1:18.1.3-1ubuntu1) ...
Selecting previously unselected package libclang-cpp18.
Preparing to unpack .../03-libclang-cpp18_1%3a18.1.3-1ubuntu1_amd64.deb ...
Unpacking libclang-cpp18 (1:18.1.3-1ubuntu1) ...
Selecting previously unselected package

In [ ]:
from google.colab import files
import json
import os

# ============================================================
# PASSO 1 — Upload dos arquivos JSON
# ============================================================
!rm -f esbmc_*.json   # apaga os de uma execução anterior: senão o Colab salva os novos como "esbmc_x (1).json" e o resto do notebook leria os antigos

print("Faça o upload dos arquivos JSON exportados anteriormente:")
print("(o esbmc_jsons.zip baixado pelo ML.ipynb, ou os 6 JSON soltos: esbmc_scanner, esbmc_rfp, esbmc_hdfs, esbmc_bpi12, esbmc_bpi17, esbmc_rtf)")
uploaded = files.upload()
print(f"\nArquivos recebidos: {list(uploaded.keys())}")

# se veio o ZIP, extrai os JSON (mesmo que o navegador tenha renomeado o ZIP para "esbmc_jsons (1).zip")
import zipfile
for nome in list(uploaded.keys()):
    if nome.lower().endswith('.zip'):
        with zipfile.ZipFile(nome) as z:
            z.extractall('.')

# Se o navegador/Colab renomeou algum arquivo ("esbmc_scanner (1).json"), volta ao nome esperado
import re, shutil
for nome in list(uploaded.keys()):
    base = re.sub(r' \(\d+\)(?=\.json$)', '', nome)
    if base != nome:
        shutil.move(nome, base)
print("Arquivos esbmc_*.json no ambiente:", sorted(f for f in os.listdir('.') if f.startswith('esbmc_')))

Faça o upload dos arquivos JSON exportados anteriormente:
(o esbmc_jsons.zip baixado pelo ML.ipynb, ou os 6 JSON soltos: esbmc_scanner, esbmc_rfp, esbmc_hdfs, esbmc_bpi12, esbmc_bpi17, esbmc_rtf)


Saving esbmc_bpi12.json to esbmc_bpi12.json
Saving esbmc_bpi17.json to esbmc_bpi17.json
Saving esbmc_hdfs.json to esbmc_hdfs.json
Saving esbmc_rfp.json to esbmc_rfp.json
Saving esbmc_rtf.json to esbmc_rtf.json
Saving esbmc_scanner.json to esbmc_scanner.json

Arquivos recebidos: ['esbmc_bpi12.json', 'esbmc_bpi17.json', 'esbmc_hdfs.json', 'esbmc_rfp.json', 'esbmc_rtf.json', 'esbmc_scanner.json']
Arquivos esbmc_*.json no ambiente: ['esbmc_bpi12.json', 'esbmc_bpi17.json', 'esbmc_hdfs.json', 'esbmc_rfp.json', 'esbmc_rtf.json', 'esbmc_scanner.json']


In [ ]:
# ============================================================
# PASSO 2 — Conversores e geradores de safety
# ============================================================

def gerar_safety(list_of_traces, evento_pre, evento_pos, msg):
    """Gera código Python com assert de safety genérico."""
    lines = ["def testar_traces():"]
    for i, trace in enumerate(list_of_traces):
        lines.append(f"    # Trace {i}")
        lines.append("    houve_pre = 0")
        for ev in trace:
            if ev == evento_pre:
                lines.append("    houve_pre = 1")
            elif ev == evento_pos:
                lines.append(f"    assert houve_pre == 1, '{msg}'")
    lines += ["\nif __name__ == '__main__':", "    testar_traces()"]
    return "\n".join(lines)

# Scanner: payer nunca sem scanner
def converter_scanner(trace):
    res = []
    for ev in trace:
        ev = str(ev).strip().lower()
        if 'scanner' in ev:       res.append(1)
        elif 'payer' in ev:       res.append(2)
        else:                     res.append(0)
    return res

# BPI RfP: payment_handled nunca sem final_approved
def converter_rfp(trace):
    res = []
    for ev in trace:
        ev = str(ev).strip().lower()
        if 'final_approved' in ev:    res.append(1)
        elif 'payment_handled' in ev: res.append(2)
        else:                         res.append(0)
    return res

# HDFS: "Received block" (E6 ou E9) nunca sem "Receiving block" (E5) antes (templates do LogHub; ver seção 10 do ML.ipynb)
def converter_hdfs(trace):
    res = []
    for ev in trace:
        ev = str(ev).strip().upper()
        if ev == 'E5':               res.append(1)
        elif ev in ('E6', 'E9'):     res.append(2)
        else:                        res.append(0)
    return res

# BPI 2012: offer_accepted nunca sem app_accepted
def converter_bpi12(trace):
    res = []
    for ev in trace:
        ev = str(ev).strip().lower()
        if 'app_accepted' in ev:    res.append(1)
        elif 'offer_accepted' in ev: res.append(2)
        else:                        res.append(0)
    return res

# BPI 2017: offer_accepted nunca sem app_accepted
def converter_bpi17(trace):
    res = []
    for ev in trace:
        ev = str(ev).strip().lower()
        if 'app_accepted' in ev:    res.append(1)
        elif 'offer_accepted' in ev: res.append(2)
        else:                        res.append(0)
    return res

# Road Traffic Fine: Payment nunca sem Create Fine
def converter_rtf(trace):
    res = []
    for ev in trace:
        ev = str(ev).strip().lower()
        if 'create fine' in ev:  res.append(1)
        elif 'payment' in ev:    res.append(2)
        else:                    res.append(0)
    return res

# ============================================================
# PASSO 3 — Gera os arquivos .py para o ESBMC
# ============================================================

configs = [
    ("esbmc_scanner.json", converter_scanner, "SAFETY: payer sem scanner",          "scanner"),
    ("esbmc_rfp.json",     converter_rfp,     "SAFETY: payment sem final_approved",  "rfp"),
    ("esbmc_hdfs.json",    converter_hdfs,    "SAFETY: Received block sem Receiving block", "hdfs"),
    ("esbmc_bpi12.json",   converter_bpi12,   "SAFETY: offer_accepted sem app_accepted", "bpi12"),
    ("esbmc_bpi17.json",   converter_bpi17,   "SAFETY: offer_accepted sem app_accepted", "bpi17"),
    ("esbmc_rtf.json",     converter_rtf,     "SAFETY: payment sem create_fine",     "rtf"),
]

for arquivo, conversor, msg, sigla in configs:
    if not os.path.exists(arquivo):
        print(f"⚠️  {arquivo} não encontrado — pulando")
        continue

    with open(arquivo, "r", encoding="utf-8") as f:
        dados = json.load(f)

    def unicas(seqs):
        # traces com a mesma conversão geram exatamente os mesmos asserts: verificar cada padrão uma vez
        # basta e dá o mesmo resultado (o Road Traffic Fine tem 150 mil traces, mas só ~90 padrões)
        return [list(p) for p in dict.fromkeys(tuple(conversor(s)) for s in seqs)]

    aalpy_num   = unicas(dados["aalpy_sequences"])
    agilkia_num = unicas(dados["agilkia_sequences"])

    nome_aalpy   = f"check_aalpy_{sigla}.py"
    nome_agilkia = f"check_agilkia_{sigla}.py"

    with open(nome_aalpy, "w") as f:
        f.write(gerar_safety(aalpy_num, 1, 2, msg))

    with open(nome_agilkia, "w") as f:
        f.write(gerar_safety(agilkia_num, 1, 2, msg))

    ml_num = unicas(dados.get("ml_sequences", []))
    if ml_num:
        with open(f"check_ml_{sigla}.py", "w") as f:
            f.write(gerar_safety(ml_num, 1, 2, msg))
    else:
        print(f"⚠️  {arquivo} sem 'ml_sequences' — exporte de novo pelo notebook principal (subseção 12.1)")

    print(f"✅ {sigla} — padrões únicos: AALpy {len(aalpy_num)}/{len(dados['aalpy_sequences'])} | "
          f"Agilkia {len(agilkia_num)}/{len(dados['agilkia_sequences'])} | ML {len(ml_num)}/{len(dados.get('ml_sequences', []))}")

print("\nArquivos gerados. Agora rode o ESBMC.")

✅ scanner — padrões únicos: AALpy 252/295 | Agilkia 108/4818 | ML 65/68
✅ rfp — padrões únicos: AALpy 34/136 | Agilkia 42/6886 | ML 24/40
✅ hdfs — padrões únicos: AALpy 579/1069 | Agilkia 881/15000 | ML 81/87
✅ bpi12 — padrões únicos: AALpy 163/2473 | Agilkia 1015/13087 | ML 24/49
✅ bpi17 — padrões únicos: AALpy 98/4146 | Agilkia 414/31509 | ML 29/89
✅ rtf — padrões únicos: AALpy 48/206 | Agilkia 90/150370 | ML 27/39

Arquivos gerados. Agora rode o ESBMC.


In [ ]:
# ============================================================
# PASSO 4 — Roda o ESBMC
# ============================================================

!esbmc --version | head -n 1   # se isto não mostrar a versão, o binário não foi instalado (sem isto, o erro de um binário ausente aparece como saída vazia)

!echo "=== SCANNER — AALpy ===" && esbmc check_aalpy_scanner.py 2>&1 | grep -iE "VERIFICATION|Generated|FAILED|error"
!echo "=== SCANNER — Agilkia ===" && esbmc check_agilkia_scanner.py 2>&1 | grep -iE "VERIFICATION|Generated|FAILED|error"

!echo "=== BPI RfP — AALpy ===" && esbmc check_aalpy_rfp.py 2>&1 | grep -iE "VERIFICATION|Generated|FAILED|error"
!echo "=== BPI RfP — Agilkia ===" && esbmc check_agilkia_rfp.py 2>&1 | grep -iE "VERIFICATION|Generated|FAILED|error"

!echo "=== HDFS — AALpy ===" && esbmc check_aalpy_hdfs.py 2>&1 | grep -iE "VERIFICATION|Generated|FAILED|error"
!echo "=== HDFS — Agilkia ===" && esbmc check_agilkia_hdfs.py 2>&1 | grep -iE "VERIFICATION|Generated|FAILED|error"

!echo "=== BPI 2012 — AALpy ===" && esbmc check_aalpy_bpi12.py 2>&1 | grep -iE "VERIFICATION|Generated|FAILED|error"
!echo "=== BPI 2012 — Agilkia ===" && esbmc check_agilkia_bpi12.py 2>&1 | grep -iE "VERIFICATION|Generated|FAILED|error"

!echo "=== BPI 2017 — AALpy ===" && esbmc check_aalpy_bpi17.py 2>&1 | grep -iE "VERIFICATION|Generated|FAILED|error"
!echo "=== BPI 2017 — Agilkia ===" && esbmc check_agilkia_bpi17.py 2>&1 | grep -iE "VERIFICATION|Generated|FAILED|error"

!echo "=== Road Traffic Fine — AALpy ===" && esbmc check_aalpy_rtf.py 2>&1 | grep -iE "VERIFICATION|Generated|FAILED|error"
!echo "=== Road Traffic Fine — Agilkia ===" && esbmc check_agilkia_rtf.py 2>&1 | grep -iE "VERIFICATION|Generated|FAILED|error"

# Sequências geradas pelo ML (chave ml_sequences dos JSON)
!echo "=== SCANNER — ML ===" && esbmc check_ml_scanner.py 2>&1 | grep -iE "VERIFICATION|Generated|FAILED|error"
!echo "=== BPI RfP — ML ===" && esbmc check_ml_rfp.py 2>&1 | grep -iE "VERIFICATION|Generated|FAILED|error"
!echo "=== HDFS — ML ===" && esbmc check_ml_hdfs.py 2>&1 | grep -iE "VERIFICATION|Generated|FAILED|error"
!echo "=== BPI 2012 — ML ===" && esbmc check_ml_bpi12.py 2>&1 | grep -iE "VERIFICATION|Generated|FAILED|error"
!echo "=== BPI 2017 — ML ===" && esbmc check_ml_bpi17.py 2>&1 | grep -iE "VERIFICATION|Generated|FAILED|error"
!echo "=== Road Traffic Fine — ML ===" && esbmc check_ml_rtf.py 2>&1 | grep -iE "VERIFICATION|Generated|FAILED|error"

ESBMC version 8.6.0 64-bit x86_64 linux
=== SCANNER — AALpy ===
Generated 11 VCC(s), 0 remaining after simplification (0 assignments)
** 0 of 11 properties failed, 11 passed
VERIFICATION SUCCESSFUL
=== SCANNER — Agilkia ===
Generated 89 VCC(s), 0 remaining after simplification (0 assignments)
** 0 of 89 properties failed, 89 passed
VERIFICATION SUCCESSFUL
=== BPI RfP — AALpy ===
Generated 3 VCC(s), 0 remaining after simplification (0 assignments)
** 0 of 3 properties failed, 3 passed
VERIFICATION SUCCESSFUL
=== BPI RfP — Agilkia ===
Generated 25 VCC(s), 0 remaining after simplification (0 assignments)
** 0 of 25 properties failed, 25 passed
VERIFICATION SUCCESSFUL
=== HDFS — AALpy ===
Generated 1253 VCC(s), 0 remaining after simplification (0 assignments)
** 0 of 1253 properties failed, 1253 passed
VERIFICATION SUCCESSFUL
=== HDFS — Agilkia ===
Generated 2786 VCC(s), 0 remaining after simplification (0 assignments)
** 0 of 2786 properties failed, 2786 passed
VERIFICATION SUCCESSFUL
===

## Verificação da própria máquina de estados (MEF)

As verificações acima checam **listas de sequências**. Aqui o alvo é o **modelo aprendido**: cada JSON traz a
chave "mef" (tabela de transições e saídas do L*), e a propriedade de safety é verificada em **todas as
execuções do modelo** a partir do estado inicial, usando só transições reais (saída diferente de nao_observado).

- **Busca em Python (referência):** alcançabilidade no produto (estado da MEF × "já houve o evento antes").
  É exata e cobre execuções de qualquer tamanho. Também informa se o evento "depois" é alcançável no modelo:
  se não for, a propriedade vale de forma **vazia**.
- **ESBMC (BMC limitado):** gera check_mef_<sigla>.py, que codifica a máquina e deixa o ESBMC escolher as
  entradas (nondet_int) por K_ESBMC passos. É a versão "formal" da mesma checagem e deve concordar com a busca
  em Python dentro desse limite.

Precisa dos JSON exportados depois da troca do oráculo (que trazem "mef").

**Limite de tempo:** cada dataset roda com no máximo 300 s, com 8 passos (--unwind 9). Nos modelos grandes (BPI 2012 e BPI 2017, com mais de mil
estados) a codificação do ESBMC tem dezenas de milhares de casos por passo e pode estourar o limite; nesse caso vale o
resultado da busca em Python, que é exata, e a verificação com ESBMC fica registrada como não concluída.

**Codificação:** o programa só escreve as transições reais da máquina; as artificiais (saída nao_observado, de 80% a 92% das transições)
caem no "return -1" final e são descartadas pela suposição prox >= 0, o que não muda o resultado e reduz o programa de 6 a 12 vezes.
A última célula mostra o final dos logs do Scanner e do RfP, para ver em que fase o ESBMC estava ao estourar o tempo.

**Modelos aproximados:** no HDFS, no BPI 2012 e no BPI 2017, o L* parou no limite de tempo e o modelo é aproximado. A busca em Python e as sequências do AALpy desses datasets valem para o modelo parcial. No Scanner, no BPI RfP e no Road Traffic Fine o modelo é completo.

**Propriedade do HDFS:** os eventos são códigos de template do LogHub. A propriedade é "Received block" (E6 ou E9) nunca sem "Receiving block" (E5) antes. Com três réplicas ela só exige ao menos um E5 antes, sem contar um por DataNode.

In [ ]:
from collections import deque

K_ESBMC = 8   # passos explorados pelo ESBMC (BMC limitado); a busca em Python cobre execuções de qualquer tamanho


def verificar_mef_python(mef, conversor):
    """Alcançabilidade no produto (estado da MEF x já houve o evento 'antes'), só por transições reais."""
    alfabeto = mef['alfabeto']
    trans    = mef['transicoes']
    classe   = {a: conversor([a])[0] for a in alfabeto}   # 1 = evento "antes", 2 = evento "depois", 0 = outro
    inicio   = (mef['inicial'], 0)
    pai      = {inicio: None}
    fila     = deque([inicio])
    ocorrencias_pos, violacao = 0, None
    while fila and violacao is None:
        est, pre = fila.popleft()
        for a in alfabeto:
            prox, saida = trans[est][a]
            if saida == 'nao_observado':
                continue
            c = classe[a]
            if c == 2:
                ocorrencias_pos += 1
                if not pre:
                    violacao = ((est, pre), a)
                    break
            novo = (prox, 1 if (pre or c == 1) else 0)
            if novo not in pai:
                pai[novo] = ((est, pre), a)
                fila.append(novo)
    caminho = None
    if violacao:
        no, a = violacao
        caminho = [a]
        while pai[no] is not None:
            no, simbolo = pai[no]
            caminho.append(simbolo)
        caminho.reverse()
    return {
        'alcancaveis'    : len({e for e, _ in pai}),
        'ocorrencias_pos': ocorrencias_pos,
        'violacao'       : violacao is not None,
        'contraexemplo'  : caminho,
    }


def gerar_esbmc_mef(mef, conversor, msg, k):
    """Programa Python para o ESBMC: a máquina codificada + entradas nondeterminísticas por k passos."""
    alfabeto = mef['alfabeto']
    classe   = [conversor([a])[0] for a in alfabeto]
    L = ["def passo(estado: int, a: int) -> int:"]
    for est, linha in enumerate(mef['transicoes']):
        # só as transições reais: as artificiais (nao_observado, 80% a 92% do modelo) caem no "return -1" final
        reais = [(i, linha[a][0]) for i, a in enumerate(alfabeto) if linha[a][1] != 'nao_observado']
        if not reais:
            continue
        L.append(f"    if estado == {est}:")
        for i, prox in reais:
            L.append(f"        if a == {i}:")
            L.append(f"            return {prox}")
    L += ["    return -1", "", "def classe(a: int) -> int:"]
    for i, c in enumerate(classe):
        if c:
            L += [f"    if a == {i}:", f"        return {c}"]
    L += ["    return 0", "", "def verificar():",
          f"    estado: int = {mef['inicial']}",
          "    houve_pre: int = 0",
          f"    for _ in range({k}):",
          "        a: int = nondet_int()",
          f"        __ESBMC_assume(a >= 0 and a < {len(alfabeto)})",
          "        prox: int = passo(estado, a)",
          "        __ESBMC_assume(prox >= 0)",
          "        c: int = classe(a)",
          "        if c == 1:",
          "            houve_pre = 1",
          "        if c == 2:",
          f"            assert houve_pre == 1, '{msg}'",
          "        estado = prox",
          "", "if __name__ == '__main__':", "    verificar()"]
    return "\n".join(L)


resultados_mef = {}
for arquivo, conversor, msg, sigla in configs:
    if not os.path.exists(arquivo):
        print(f"⚠️  {arquivo} não encontrado — pulando")
        continue
    with open(arquivo, "r", encoding="utf-8") as f:
        dados = json.load(f)
    if 'mef' not in dados:
        print(f"⚠️  {arquivo} sem 'mef' — reexporte pelo notebook principal (seção 10)")
        continue
    mef = dados['mef']
    r = resultados_mef[sigla] = verificar_mef_python(mef, conversor)
    if r['violacao']:
        veredito = 'VIOLA: ' + ' > '.join(r['contraexemplo'])
    elif r['ocorrencias_pos'] == 0:
        veredito = 'vale, mas VAZIA (o evento "depois" não é alcançável no modelo)'
    else:
        veredito = f"vale em qualquer execução ({r['ocorrencias_pos']} pontos de checagem)"
    print(f"{sigla:<8} estados alcançáveis: {r['alcancaveis']:>4}/{2 * len(mef['transicoes'])} (estado x flag) | {veredito}")
    with open(f"check_mef_{sigla}.py", "w") as f:
        f.write(gerar_esbmc_mef(mef, conversor, msg, K_ESBMC))

print("\nProgramas check_mef_*.py gerados. Rode a célula seguinte (ESBMC).")

scanner  estados alcançáveis:  228/456 (estado x flag) | vale em qualquer execução (12 pontos de checagem)
rfp      estados alcançáveis:   78/156 (estado x flag) | vale em qualquer execução (3 pontos de checagem)
hdfs     estados alcançáveis:  667/1334 (estado x flag) | vale em qualquer execução (214 pontos de checagem)
bpi12    estados alcançáveis:  965/1930 (estado x flag) | vale em qualquer execução (181 pontos de checagem)
bpi17    estados alcançáveis: 1589/3178 (estado x flag) | vale em qualquer execução (468 pontos de checagem)
rtf      estados alcançáveis:   95/190 (estado x flag) | vale em qualquer execução (52 pontos de checagem)

Programas check_mef_*.py gerados. Rode a célula seguinte (ESBMC).


In [ ]:
%%bash
# Mata esbmc que tenha ficado em segundo plano (interromper uma célula no Colab não encerra o processo filho).
# Cada dataset tem 300 s; a saída vai para arquivo (sem pipe) e o kill é forçado. K_ESBMC = 8 -> --unwind 9.
pkill -9 -f esbmc || true
for d in scanner rfp rtf hdfs bpi12 bpi17; do
  echo "=== $d — MEF ==="
  timeout -s KILL 300 esbmc check_mef_$d.py --unwind 9 > saida_mef_$d.txt 2>&1
  grep -iE "VERIFICATION|Generated|FAILED|error" saida_mef_$d.txt || echo "(sem resultado: estourou 300 s ou falhou; veja saida_mef_$d.txt)"
  pkill -9 -f esbmc || true
done

=== scanner — MEF ===
Generated 9 VCC(s), 8 remaining after simplification (6170 assignments)
** 0 of 2 properties failed, 2 passed
VERIFICATION SUCCESSFUL
=== rfp — MEF ===
Generated 9 VCC(s), 8 remaining after simplification (2572 assignments)
** 0 of 2 properties failed, 2 passed
VERIFICATION SUCCESSFUL
=== rtf — MEF ===
Generated 9 VCC(s), 8 remaining after simplification (3669 assignments)
** 0 of 2 properties failed, 2 passed
VERIFICATION SUCCESSFUL
=== hdfs — MEF ===
(sem resultado: estourou 300 s ou falhou; veja saida_mef_hdfs.txt)
=== bpi12 — MEF ===
(sem resultado: estourou 300 s ou falhou; veja saida_mef_bpi12.txt)
=== bpi17 — MEF ===
(sem resultado: estourou 300 s ou falhou; veja saida_mef_bpi17.txt)


bash: line 4:  6153 Killed                  timeout -s KILL 300 esbmc check_mef_$d.py --unwind 9 > saida_mef_$d.txt 2>&1
bash: line 4:  7351 Killed                  timeout -s KILL 300 esbmc check_mef_$d.py --unwind 9 > saida_mef_$d.txt 2>&1
bash: line 4:  8545 Killed                  timeout -s KILL 300 esbmc check_mef_$d.py --unwind 9 > saida_mef_$d.txt 2>&1


In [ ]:
!tail -n 6 saida_mef_scanner.txt; echo ---; tail -n 6 saida_mef_rfp.txt; echo ---; tail -n 6 saida_mef_hdfs.txt

  PASSED       [verificar.assertion.2]  line 886  assertion houve_pre == 1: SAFETY: payer sem scanner

** 0 of 2 properties failed, 2 passed

VERIFICATION SUCCESSFUL
ESBMC version 8.6.0 64-bit x86_64 linux
---
  PASSED       [verificar.assertion.2]  line 372  assertion houve_pre == 1: SAFETY: payment sem final_approved

** 0 of 2 properties failed, 2 passed

VERIFICATION SUCCESSFUL
ESBMC version 8.6.0 64-bit x86_64 linux
---
Target: 64-bit little-endian x86_64-unknown-linux with esbmclibc
Parsing check_mef_hdfs.py
Converting
